In [1]:
# === Imports ===

from dataclasses import dataclass, field
from typing import Tuple, Type

import numpy as np
from matplotlib import pyplot as plt
from matplotlib import style as mplstyle
from numba import jit
from scipy.fft import next_fast_len
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import LinearOperator, lsmr

mplstyle.use("./docs/pyscopee.mplstyle")

%matplotlib widget

In [2]:
# the regularly sampled signal is loaded
data_regular = np.loadtxt("signal_regular.txt", delimiter=",", skiprows=1)

t_values_regular = data_regular[:, 0]
y_values_regular = data_regular[:, 1]

In [ ]:
# === Models ===


@dataclass
class Bandlimits:
    """
    A dataclass that contains the lower and upper bandlimit frequencies of the Fourier
    Transform.

    It also holds a flag whether the lower bandlimit is zero and an effective lower
    bandlimit that is used for comparisons where a lower bandlimit of zero has to be
    excluded from the consideration.

    """

    low: float
    high: float
    nonzero_low: float

    low_is_zero: bool = field(init=False)

    def __post_init__(self) -> None:
        """
        Validates the bandlimit frequencies.

        """

        if self.low >= self.high:
            raise ValueError(
                f"The lower bandlimit {self.low:.5e} exceeds the upper bandlimit "
                f"{self.high:.5e}."
            )

        if self.low < 0.0:
            raise ValueError(f"The lower bandlimit {self.low:.5e} is less than zero.")

        self.low_is_zero = self.low == 0.0

        # the effective lower bandlimit is not set yet because this requires the
        # frequencies of the Fourier Transform


@dataclass
class FourierSpecs:
    """
    A dataclass that contains

    - the positive frequencies of the Fourier Transform (negative frequencies are
        given by the Hermitian symmetry of the Fourier Transform for real-valued
        signals)
    - the lower and upper bandlimit
    - whether the lower bandlimit is zero
    - the effective lower bandlimit
    - the size of the coefficient vector x for the Fourier Transform

    """

    freqs: np.ndarray
    bandlimits: Bandlimits
    freq_indices_in_nonzero_bandlimit_from: int = field(init=False)
    freq_indices_in_nonzero_bandlimit_to: int = field(init=False)
    freq_indices_zero_coeffs_below_low_to: int = field(init=False)
    x_vect_size: int = field(init=False)
    x_real_imag_split_index: int = field(init=False)

    def __post_init__(self) -> None:
        """
        Evaluates where the real and imaginary coefficient of the Fourier coefficients
        are located within the bandlimit at the positive frequencies of the Fourier
        Transform.

        Besides, it also calculates the size of the coefficient vector x for the Fourier
        Transform and how the mapping between x and the real and imaginary parts of the
        Fourier coefficients is done.

        """

        # the indices of the coefficients that are located at all NONZERO frequencies
        # within the bandlimit are found
        self.freq_indices_in_nonzero_bandlimit_from = int(
            np.searchsorted(a=self.freqs, v=self.bandlimits.nonzero_low, side="left")
        )
        self.freq_indices_in_nonzero_bandlimit_to = int(
            np.searchsorted(a=self.freqs, v=self.bandlimits.high, side="right")
        )

        # for the indices of the coefficients below the lower bandlimit, there has to be
        # a distinction between the case when the lower bandlimit is zero and when it
        # is not
        self.freq_indices_zero_coeffs_below_low_to = (
            self.freq_indices_in_nonzero_bandlimit_from
            if not self.bandlimits.low_is_zero
            else 0
        )

        # for each nonzero positive frequency, there are 2 coefficients (real and
        # imaginary part);
        # for a potentially zero lower bandlimit, there has to be an additional
        # real coefficient for the zero frequency;
        # this is also accounted for in the split between the real and imaginary
        # coefficients in x
        if not self.bandlimits.low_is_zero:
            x_vect_size_offset = 0
        else:
            x_vect_size_offset = 1

        num_freqs_in_nonzero_bandlimit = (
            self.freq_indices_in_nonzero_bandlimit_to
            - self.freq_indices_in_nonzero_bandlimit_from
        )
        self.x_real_imag_split_index = (
            x_vect_size_offset + num_freqs_in_nonzero_bandlimit
        )
        self.x_vect_size = x_vect_size_offset + 2 * num_freqs_in_nonzero_bandlimit

        return

    def __len__(self) -> int:
        """
        Returns the number of samples in the time domain which is equal to the size of
        the full frequencies of the Fourier Transform.

        """

        return self.freqs.size


# === Functions ===


def prepare_signal_for_fast_fft(
    t_grid: np.ndarray,
    y_values: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Prepares the signal for a fast Fourier Transform by interpolating it to a
    regularly spaced grid with the size of the next fast length for real-valued
    Fourier Transforms.

    Parameters
    ----------
    t_grid, y_values : :class:`numpy.ndarray` of shape (m,)
        The time vector and the corresponding signal values.

    Returns
    -------
    t_grid_fast, y_values_fast : :class:`numpy.ndarray` of shape (n,)
        The regularly spaced time vector and the corresponding signal values for a fast
        Fourier Transform.

    """

    n_rfft_fast = next_fast_len(t_grid.size, real=True)
    t_grid_fast = np.linspace(start=t_grid[0], stop=t_grid[-1], num=n_rfft_fast,)

    return t_grid_fast, np.interp(x=t_grid_fast, xp=t_grid, fp=y_values)

def get_fourier_specs(
    t_grid: np.ndarray,
    bandlimit_low: float,
    bandlimit_high,
) -> FourierSpecs:
    """
    Computes the Fourier Transform specifications for the given time vector and
    bandlimit frequencies.

    Parameters
    ----------
    t_grid : :class:`numpy.ndarray`
        The time vector of equidistant time points at which the signal is sampled.
    bandlimit_low, bandlimit_high : :class:`float`
        The lower and upper bandlimit frequencies of the Fourier Transform.

    Returns
    -------
    fourier_specs : :class:`FourierSpecs`
        The Fourier Transform specifications for the given time vector and bandlimit
        frequencies. Please refer to the class :class:`FourierSpecs` for more details.

    Raises
    ------
    ValueError
        If there are no frequencies located between the lower and upper bandlimit.
    ValueError
        If the upper bandlimit does not lead to the exclusion of the Nyquist frequency.

    """

    # the full and positive frequencies of the Fourier Transform are computed after the
    # size for a really fast real Fourier Transform is determined
    delta_t = (t_grid[-1] - t_grid[0]) / (t_grid.size - 1)
    freqs = np.fft.rfftfreq(n=t_grid.size, d=delta_t)  # type: ignore

    # the bandlimit frequencies are validated for themselves
    # NOTE: the effective lower bandlimit is clipped to 0.5 times the first positive
    #       frequency to exclude a potentially zero bandlimit frequency from all related
    #       comparisons
    bandlimits = Bandlimits(
        low=bandlimit_low,
        high=bandlimit_high,
        nonzero_low=max(0.5 * freqs[1], bandlimit_low),
    )

    # next, the bandlimits need to be validated against the frequencies of the
    # Fourier Transform
    # then, it is checked if there are any frequencies within the bandlimit
    if not np.any((bandlimits.low <= freqs) & (freqs <= bandlimits.high)):
        raise ValueError(
            f"There are no frequencies within the bandlimit "
            f"[{bandlimits.low:.5e}, {bandlimits.high:.5e}]."
        )

    # afterwards, the upper bandlimit is checked against the Nyquist frequency which
    # should be excluded
    if bandlimits.high >= freqs[-1]:
        raise ValueError(
            f"The upper bandlimit {bandlimits.high:.5e} does not lead to the exclusion "
            f"of the Nyquist frequency {freqs[-1]:.5e}."
        )

    # finally, the Fourier Transform specifications are returned
    return FourierSpecs(
        freqs=freqs,
        bandlimits=bandlimits,
    )


def convert_x_to_real_ift_coefficients(
    x: np.ndarray,
    num_freqs: int,
    x_real_imag_split_index: int,
    bandlimit_low_is_zero: bool,
    freq_indices_in_nonzero_bandlimit_from: int,
    freq_indices_in_nonzero_bandlimit_to: int,
    freq_indices_zero_coeffs_below_low_to: int,
) -> np.ndarray:
    """
    Converts the compressed coefficients x for the Fourier Transform to the real and
    imaginary coefficients of the Inverse Fourier Transform.

    Parameters
    ----------
    x : :class:`numpy.ndarray` of shape (p,)
        The coefficients for the Fourier Transform stored in compressed form as real
        values.
    num_freqs : :class:`int`
        The number of frequencies of the Fourier Transform.
    x_real_imag_split_index : :class:`int`
        The index at which the real and imaginary coefficients are split in the
        coefficient vector x.
        So, the real coefficients can be found at ``x[0:x_real_imag_split_index]``
        while the imaginary coefficients are located at ``x[x_real_imag_split_index:]``.
    bandlimit_low_is_zero : :class:`bool`
        A flag that indicates whether the lower bandlimit is zero (``True``) or not
        (``False``).
    freq_indices_in_nonzero_bandlimit_from, freq_indices_in_nonzero_bandlimit_to : :class:`int`
        The index from the first to the last positive nonzero frequency within the
        bandlimit. For obtaining the nonzero frequencies within the bandlimit, these
        indices can be used as ``freqs[freq_indices_in_nonzero_bandlimit_from:freq_indices_in_nonzero_bandlimit_to]``.
    freq_indices_zero_coeffs_below_low_to : :class:`int`
        The index up to which the frequencies lie below the lower bandlimit (if any).
        For obtaining the frequencies below the lower bandlimit, these indices can be
        used as ``freqs[0:freq_indices_zero_coeffs_below_low_to]``.

    Returns
    -------
    x_real_ift : :class:`numpy.ndarray` of shape (floor(m/2) + 1,)
        The real coefficients of the Inverse Fourier Transform.

    """

    # the result is initialised as an empty Array because filling it fully with zeros
    # is not necessary
    x_real_ift = np.empty(shape=(num_freqs,), dtype=np.complex128)

    # if the lower bandlimit is not zero, the leading zeros are filled into the result
    if not bandlimit_low_is_zero:
        x_offset_index = 0
        x_real_ift[0:freq_indices_zero_coeffs_below_low_to] = 0.0

    # otherwise, if the lower bandlimit is zero, the first coefficient is the real
    # coefficient for the zero frequency
    else:
        x_offset_index = 1
        x_real_ift[0] = x[0]

    # the real and imaginary coefficients are split and stored in the result
    x_real_ift[
        freq_indices_in_nonzero_bandlimit_from:freq_indices_in_nonzero_bandlimit_to
    ] = (
        x[x_offset_index:x_real_imag_split_index]
        + 1.0j * x[x_real_imag_split_index : x.size]
    )

    # finally, the trailing zeros are filled into the result
    x_real_ift[freq_indices_in_nonzero_bandlimit_to : x_real_ift.size] = 0.0

    return x_real_ift


def compress_real_ift_coefficients_to_xlike(
    x_real_ift: np.ndarray,
    x_vect_size: int,
    x_real_imag_split_index: int,
    bandlimit_low_is_zero: bool,
    freq_indices_in_nonzero_bandlimit_from: int,
    freq_indices_in_nonzero_bandlimit_to: int,
) -> np.ndarray:
    """
    Compresses the real and imaginary coefficients of the Inverse Fourier Transform to
    a compressed form as real values similar (but not fully identical) to the
    real x-coefficients of the Fourier Transform.

    For the parameters, please refer to the function :func:`convert_x_to_real_ift_coefficients`.

    """  # noqa: E501

    # the result is initialised as an empty Array because it will not have a single
    # zero value
    x = np.empty(shape=(x_vect_size,), dtype=np.float64)

    # if the lower bandlimit is zero, the first coefficient is the real coefficient for
    # the zero frequency
    x_offset_index = 0
    if bandlimit_low_is_zero:
        x[0] = 2.0 * x_real_ift.real[0]
        x_offset_index = 1

    # then, the twofold of the real coefficients are stored in the first half of x
    x[x_offset_index:x_real_imag_split_index] = (
        x_real_ift[
            freq_indices_in_nonzero_bandlimit_from:freq_indices_in_nonzero_bandlimit_to
        ].real
    )

    # finally, the negative imaginary coefficients are stored in the second half of x
    x[x_real_imag_split_index : x.size] = (
        x_real_ift[
            freq_indices_in_nonzero_bandlimit_from:freq_indices_in_nonzero_bandlimit_to
        ].imag
    )

    return x


def second_order_differences(y: np.ndarray) -> np.ndarray:
    """
    Computes the second-order differences of the given vector y.

    Parameters
    ----------
    y : :class:`numpy.ndarray` of shape (m,)
        The data for which the second-order differences are computed.

    Returns
    -------
    y_diff : :class:`numpy.ndarray` of shape (m-2,)
        The second-order differences of ``y``.

    """

    y_diff = np.empty(shape=(y.size - 2,), dtype=np.float64)
    for i in range(y_diff.size):
        y_diff[i] = y[i + 2] - 2.0 * y[i + 1] + y[i]

    return y_diff


def second_order_differences_transposed(y: np.ndarray) -> np.ndarray:
    """
    Computes the second order differences of the given array y when the finite
    difference matrix ``D`` in the product is transposed.
    This is equivalent to padding ``y`` with 1 leading and trailing zeros and applying
    the second order differences with flipped coefficients (has no effect in this case).

    Parameters
    ----------
    y : :class:`numpy.ndarray` of shape (m,)
        The data for which the second-order differences are computed.

    Returns
    -------
    y_diff : :class:`numpy.ndarray` of shape (m,)
        The second-order differences of ``y``.

    """

    y_diff = np.empty(shape=(y.size,), dtype=np.float64)
    y_diff[0] = -2.0 * y[0] + y[1]
    for i in range(1, y_diff.size - 1):
        y_diff[i] = y[i - 1] - 2.0 * y[i] + y[i + 1]

    y_diff[y_diff.size - 1] = y[y.size - 2] - 2.0 * y[y.size - 1]

    return y_diff


# === Classes ===


class BandlimitedIFT(LinearOperator):
    """
    A class that represents a linear operator for the Inverse Fourier Transform of a
    bandlimited signal whose Fourier Transform is stored in a compressed form.

    """

    def __init__(
        self,
        t_grid: np.ndarray,
        bandlimit_high: float,
        bandlimit_low: float = 0.0,
    ) -> None:

        self.original_size: int = t_grid.size
        self.fourier_specs: FourierSpecs = get_fourier_specs(
            t_grid=t_grid,
            bandlimit_low=bandlimit_low,
            bandlimit_high=bandlimit_high,
        )

        self.shape: Tuple[int, int] = (
            t_grid.size,
            self.fourier_specs.x_vect_size,
        )
        self.dtype: Type = np.float64

    def _matvec(self, x: np.ndarray) -> np.ndarray:
        return np.fft.irfft(
            convert_x_to_real_ift_coefficients_jit(
                x=x,
                num_freqs=len(self.fourier_specs),
                x_real_imag_split_index=self.fourier_specs.x_real_imag_split_index,
                bandlimit_low_is_zero=self.fourier_specs.bandlimits.low_is_zero,
                freq_indices_in_nonzero_bandlimit_from=self.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
                freq_indices_in_nonzero_bandlimit_to=self.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
                freq_indices_zero_coeffs_below_low_to=self.fourier_specs.freq_indices_zero_coeffs_below_low_to,
            ),
            n=self.original_size,
        )

    def _rmatvec(self, x: np.ndarray) -> np.ndarray:
        return compress_real_ift_coefficients_to_xlike_jit(
            x_real_ift=np.fft.irfft(x, n = self.original_size),
            x_vect_size=self.fourier_specs.x_vect_size,
            x_real_imag_split_index=self.fourier_specs.x_real_imag_split_index,
            bandlimit_low_is_zero=self.fourier_specs.bandlimits.low_is_zero,
            freq_indices_in_nonzero_bandlimit_from=self.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
            freq_indices_in_nonzero_bandlimit_to=self.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
        )


convert_x_to_real_ift_coefficients_jit = jit(nopython=True)(
    convert_x_to_real_ift_coefficients
)
compress_real_ift_coefficients_to_xlike_jit = jit(nopython=True)(
    compress_real_ift_coefficients_to_xlike
)
second_order_differences_jit = jit(nopython=True)(second_order_differences)
second_order_differences_transposed_jit = jit(nopython=True)(
    second_order_differences_transposed
)

t = np.linspace(0.0, 1.0, 10)
fourier_specs = get_fourier_specs(t, 0.0, 3.0)
x = np.arange(1, fourier_specs.x_vect_size + 1).astype(np.float64)
x_real_ift = convert_x_to_real_ift_coefficients(
    x,
    len(fourier_specs),
    fourier_specs.x_real_imag_split_index,
    fourier_specs.bandlimits.low_is_zero,
    fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    fourier_specs.freq_indices_in_nonzero_bandlimit_to,
    fourier_specs.freq_indices_zero_coeffs_below_low_to,
)
x_real_ift_jit = convert_x_to_real_ift_coefficients_jit(
    x,
    len(fourier_specs),
    fourier_specs.x_real_imag_split_index,
    fourier_specs.bandlimits.low_is_zero,
    fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    fourier_specs.freq_indices_in_nonzero_bandlimit_to,
    fourier_specs.freq_indices_zero_coeffs_below_low_to,
)

x_reconstructed = compress_real_ift_coefficients_to_xlike(
    x_real_ift,
    fourier_specs.x_vect_size,
    fourier_specs.x_real_imag_split_index,
    fourier_specs.bandlimits.low_is_zero,
    fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    fourier_specs.freq_indices_in_nonzero_bandlimit_to,
)
x_reconstructed_jit = compress_real_ift_coefficients_to_xlike_jit(
    x_real_ift,
    fourier_specs.x_vect_size,
    fourier_specs.x_real_imag_split_index,
    fourier_specs.bandlimits.low_is_zero,
    fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    fourier_specs.freq_indices_in_nonzero_bandlimit_to,
)

def make_p_matrix(
    fourier_specs: FourierSpecs,
):
    p_matrix = lil_matrix((fourier_specs.__len__(), fourier_specs.x_vect_size), dtype=np.complex128)

    if fourier_specs.bandlimits.low_is_zero:
        p_matrix[0, 0] = 1.0

    num_freqs_in_nonzero_bandlimit = (
        fourier_specs.freq_indices_in_nonzero_bandlimit_to
        - fourier_specs.freq_indices_in_nonzero_bandlimit_from
    )
    for i in range(fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to):
        p_matrix[i, i] = 1.0
        p_matrix[i, num_freqs_in_nonzero_bandlimit + i] = 1.0j

    return p_matrix.tocsr()

p_matrix = make_p_matrix(fourier_specs)
print(p_matrix.toarray())
print(p_matrix.T.toarray())

print(np.allclose(p_matrix @ x, x_real_ift))
print(x_reconstructed)
print(p_matrix.T @ np.arange(1, p_matrix.shape[0] + 1))
print(np.allclose(p_matrix.T @ x_real_ift, x_reconstructed))


In [ ]:
y = np.random.rand(t.size)
y_diff = second_order_differences(y)
assert np.allclose(y_diff, np.convolve(y, [1, -2, 1], mode="valid"))
y_diff = second_order_differences_transposed(y)
assert y.size == y_diff.size
assert np.allclose(y_diff, np.convolve(np.pad(y, (1, 1)), [1, -2, 1], mode="valid"))
%timeit convert_x_to_real_ift_coefficients(x, len(fourier_specs), fourier_specs.x_real_imag_split_index, fourier_specs.bandlimits.low_is_zero, fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to, fourier_specs.freq_indices_zero_coeffs_below_low_to)
%timeit convert_x_to_real_ift_coefficients_jit(x, len(fourier_specs.freqs), fourier_specs.x_real_imag_split_index, fourier_specs.bandlimits.low_is_zero, fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to, fourier_specs.freq_indices_zero_coeffs_below_low_to)
%timeit compress_real_ift_coefficients_to_xlike(x_real_ift, fourier_specs.x_vect_size, fourier_specs.x_real_imag_split_index, fourier_specs.bandlimits.low_is_zero, fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to)
%timeit compress_real_ift_coefficients_to_xlike_jit(x_real_ift, fourier_specs.x_vect_size, fourier_specs.x_real_imag_split_index, fourier_specs.bandlimits.low_is_zero, fourier_specs.freq_indices_in_nonzero_bandlimit_from, fourier_specs.freq_indices_in_nonzero_bandlimit_to)
print("Timing derivatives")
%timeit np.convolve(np.pad(y, (2, 2)), [1, -2, 1], mode="valid")
%timeit second_order_differences(y)
%timeit second_order_differences_jit(y)
%timeit second_order_differences_transposed(y)
%timeit second_order_differences_transposed_jit(y)

In [ ]:
t_values_fast, y_values_fast = prepare_signal_for_fast_fft(
    t_grid=t_values_regular,
    y_values=y_values_regular,
)

linop = BandlimitedIFT(
    t_grid=t_values_fast,
    bandlimit_low=400.0,
    bandlimit_high=20_000.0,
    # bandlimit_high=101243.0,
)

x0 = compress_real_ift_coefficients_to_xlike_jit(
    x_real_ift=np.fft.rfft(y_values_fast),
    x_vect_size=linop.fourier_specs.x_vect_size,
    x_real_imag_split_index=linop.fourier_specs.x_real_imag_split_index,
    bandlimit_low_is_zero=linop.fourier_specs.bandlimits.low_is_zero,
    freq_indices_in_nonzero_bandlimit_from=linop.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    freq_indices_in_nonzero_bandlimit_to=linop.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
)

bandlimited_fft = lsmr(
    A=linop,
    b=y_values_fast,
    x0=x0,
    show=True,
)

# %timeit lsmr(A=linop, b=y_values_fast, x0=x0)

In [ ]:
plt.close("all")

fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()

solution = convert_x_to_real_ift_coefficients(
    x0,
    len(linop.fourier_specs),
    linop.fourier_specs.x_real_imag_split_index,
    linop.fourier_specs.bandlimits.low_is_zero,
    linop.fourier_specs.freq_indices_in_nonzero_bandlimit_from,
    linop.fourier_specs.freq_indices_in_nonzero_bandlimit_to,
    linop.fourier_specs.freq_indices_zero_coeffs_below_low_to,
)
# solution.imag = -solution.imag

signal_reconstructed = np.fft.irfft(
    solution,
    n=t_values_fast.size,
)
# signal_reconstructed = linop @ x0

ax.plot(
    np.fft.rfftfreq(n=t_values_fast.size, d=t_values_fast[1] - t_values_fast[0]),
    np.fft.rfft(y_values_fast, ).real,
    lw=4,
)
ax.plot(
    np.fft.rfftfreq(n=t_values_fast.size, d=t_values_fast[1] - t_values_fast[0]),
    solution.real,
)

ax2.plot(t_values_regular, y_values_regular, label="Original Signal")
ax2.plot(t_values_fast, signal_reconstructed, label="Reconstructed Signal")

In [ ]:
a = np.linspace(0, 1, 101)
b = np.sin(4 * np.pi * a) + np.sin(8 * np.pi * a)
nfn = next_fast_len(a.size, real=True)

a_freqs = np.fft.rfftfreq(n=a.size, d=a[1] - a[0])
a_freqs_nfn = np.fft.rfftfreq(n=nfn, d=a[1] - a[0])

b_fft = np.fft.rfft(b, n=a.size)
b_fft_nfn = np.fft.rfft(b, n=nfn)

fig, ax = plt.subplots()

ax.plot(a_freqs, np.abs(b_fft))
ax.plot(a_freqs_nfn, np.abs(b_fft_nfn))